# Exploring network traffic

This notebook queries the network traffic Delta table using Spark Connect. The Python kernel runs locally, while Spark and MinIO remain in Home Assistant.

The queries return aggregated results and avoid collecting raw packet data to the Mac.

In [ ]:
import os
import sys
from pathlib import Path

LIB = Path.cwd().parent / "jobs"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from pyspark.sql import SparkSession, functions as F

SPARK_CONNECT_URL = os.environ.get(
    "SPARK_CONNECT_URL",
    f"sc://{os.environ.get('HA_HOST', '192.168.2.180')}:{os.environ.get('SPARK_CONNECT_PORT', '15002')}",
)
SPARK_APP_NAME = os.environ.get("SPARK_APP_NAME", "HA Network Traffic Exploration")
LAKEHOUSE_ROOT = os.environ.get("LAKEHOUSE_ROOT", "s3a://lakehouse")
TABLE_PATH = f"{LAKEHOUSE_ROOT}/network_traffic/packets"

spark = (
    SparkSession.builder
    .appName(SPARK_APP_NAME)
    .remote(SPARK_CONNECT_URL)
    .getOrCreate()
)

print(f"Spark Connect: {SPARK_CONNECT_URL}")
print(f"Spark session: {SPARK_APP_NAME}")
print(f"Health check: {spark.range(1).count()} row")

## Load the Delta table

In [ ]:
traffic = spark.read.format("delta").load(TABLE_PATH)
print(f"Table: {TABLE_PATH}")
print(f"Rows: {traffic.count():,}")

In [ ]:
traffic.printSchema()

## Daily volume

In [ ]:
daily = (
    traffic.groupBy("date")
    .agg(
        F.count("*").alias("packets"),
        F.sum("length").alias("bytes"),
        F.countDistinct("src_ip").alias("sources"),
        F.countDistinct("dst_ip").alias("destinations"),
    )
    .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
    .orderBy(F.col("date").desc())
)
daily.show(30, truncate=False)

## Top sources and destinations

In [ ]:
print("Top source IPs")
(traffic.groupBy("src_ip")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("bytes").desc())
 .show(20, truncate=False))

print("Top destination IPs")
(traffic.groupBy("dst_ip")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("bytes").desc())
 .show(20, truncate=False))

## Protocols

In [ ]:
(traffic.groupBy("protocol")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("packets").desc())
 .show(30, truncate=False))

## DNS and TLS names

In [ ]:
print("Top DNS queries")
(traffic.where(F.col("dns_query").isNotNull())
 .groupBy("dns_query")
 .count()
 .orderBy(F.col("count").desc())
 .show(30, truncate=False))

print("Top TLS SNI names")
(traffic.where(F.col("tls_sni").isNotNull())
 .groupBy("tls_sni")
 .count()
 .orderBy(F.col("count").desc())
 .show(30, truncate=False))

## Hourly traffic

In [ ]:
hourly = (
    traffic.withColumn("timestamp", F.to_timestamp(F.from_unixtime("time")))
    .withColumn("hour", F.date_trunc("hour", "timestamp"))
    .groupBy("hour")
    .agg(
        F.count("*").alias("packets"),
        F.sum("length").alias("bytes"),
        F.countDistinct("src_ip").alias("sources"),
        F.countDistinct("dst_ip").alias("destinations"),
    )
    .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
    .orderBy(F.col("hour").desc())
)
hourly.show(100, truncate=False)